# 02 - M2: frozen-CLIP probes (gate: reproduce Chethan's AUROCs on the controlled protocol)
**Runtime > T4 GPU.** First run extracts CLIP features for ~10k images (~10-20 min) and caches them to Drive; later runs reuse the cache.
Gate: our AUROC point estimate must fall inside Chethan's 95% CI for each of the 6 (fold, probe) cells.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run.
import os, subprocess, base64
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
REPO = f'https://github.com/{GH_USER}/csm-clip.git'
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None

def git(*args, cwd=None):
    pre = []
    if tok:
        b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
        pre = ['-c', f'http.extraHeader=Authorization: Basic {b}']
    r = subprocess.run(['git', *pre, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(tok, '***') if tok else r.stderr
        raise RuntimeError(f'git {args[0]} failed:\n{err}')
    return r.stdout

if not os.path.exists('/content/csm-clip'):
    git('clone', '-q', REPO, '/content/csm-clip')
else:
    git('fetch', '-q', 'origin', 'main', cwd='/content/csm-clip')
    git('reset', '--hard', 'origin/main', cwd='/content/csm-clip')  # Colab copy is read-only; discards local edits
os.chdir('/content/csm-clip')
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(git('log', '--oneline', '-3'))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import numpy as np, torch
from datasets import load_dataset
from transformers import CLIPVisionModel
from src.data import load_manifest
from src.features import ClipTaps, RowDataset, extract_to_cache, FeatureStore, union_rows

PROTOCOL = 'controlled'          # the M2 gate protocol; set 'raw' for a secondary run
FOLDS = ['Midjourney', 'BigGAN', 'ADM']
manifests = [load_manifest(f) for f in FOLDS]
rev = manifests[0]['revision']; assert all(m['revision'] == rev for m in manifests)
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=rev)
clip = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14-336', torch_dtype=torch.float32).cuda()
enc = ClipTaps(clip, taps=(16, 24))
assert clip.config.num_hidden_layers == 24 and clip.config.image_size == 336
print('CLIP loaded')

In [ ]:
# Extract once (resumable: re-running skips finished chunks). train rows -> HF 'train'; test rows -> HF 'validation'.
FEAT = f'{DRIVE_ROOT}/features/{PROTOCOL}'
for split, which, hf in (('train', 'train', ds['train']), ('validation', 'test', ds['validation'])):
    rows = sorted(set(union_rows(manifests, 'train') + union_rows(manifests, 'val'))) if which == 'train' else union_rows(manifests, 'test')
    print(split, len(rows), 'rows')
    extract_to_cache(enc, RowDataset(hf, rows, PROTOCOL), f'{FEAT}/{split}', chunk=512, batch_size=32,
                     num_workers=2, device='cuda', autocast=True)

In [ ]:
# Probes on cached features (CPU, seconds). C is chosen on val (seen generators only); test is scored once.
import json, pandas as pd
from src.probes import run_fold, gate_report, gate_passes
tr, te = FeatureStore(f'{FEAT}/train'), FeatureStore(f'{FEAT}/validation')
results = [run_fold(m, tr, te, kind) for m in manifests for kind in ('univfd', 'multilayer')]
rep = pd.DataFrame(gate_report(results)); display(rep)
ok = gate_passes(rep.to_dict('records'))   # revised gate, decision D017
print('GATE PASS (revised, D017)' if ok else 'GATE FAIL: see rows with ci_overlap False or delta < -0.02')

In [ ]:
# Save results (Drive + repo runs/ folder). Then add rows to docs/experiment_log.md and commit runs/*.json.
out = {'protocol': PROTOCOL, 'revision': rev, 'results': results, 'gate': rep.to_dict('records')}
for p in (f'{DRIVE_ROOT}/runs/m2_{PROTOCOL}.json', f'/content/csm-clip/runs/m2_{PROTOCOL}.json'):
    json.dump(out, open(p, 'w'), indent=1)
print(rep[['fold','kind','ours','ours_ci','ref','ref_ci','in_ref_ci']].to_string(index=False))